<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/main2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp ipywidgets -q
!sudo apt-get install ffmpeg -qq

import os
import re
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output
from yt_dlp import YoutubeDL
from google.colab import drive

drive.mount('/content/drive')

def sanitize_name(name):
    return re.sub(r'[\\/*?:"<>|]', "", name)

def download_media(url, media_type, output_base):
    ydl_opts = {
        'format': 'bestvideo[height<=1080]+bestaudio/best[height<=1080]',
        'outtmpl': os.path.join(output_base, '%(title)s', '%(title)s.%(ext)s'),
        'merge_output_format': 'mp4',
        'writesubtitles': True,
        'subtitleslangs': ['en', 'fa'],
        'writeautomaticsub': True,
        'subtitlesformat': 'srt',
        'quiet': True,
        'no_warnings': True,
        'progress_hooks': [progress_hook],
        'postprocessors': [{
            'key': 'FFmpegVideoConvertor',
            'preferedformat': 'mp4',
        }],
    }

    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        base_path = sanitize_name(info['title']) if media_type == 'playlist' else output_base

        if media_type == 'playlist':
            ydl_opts['outtmpl'] = os.path.join(output_base, '%(playlist_title)s', '%(title)s', '%(title)s.%(ext)s')

        ydl.download([url])
        return info

def progress_hook(d):
    if d['status'] == 'downloading':
        progress.value = d['_percent_str']
        status.value = f"Downloading: {d['filename']} ({d['_speed_str']})"

def on_button_click(b):
    global output
    with output:
        clear_output()
        try:
            url = url_widget.value.strip()
            media_type = type_widget.value
            drive_path = '/content/drive/MyDrive/YT_Downloads/'

            info = download_media(url, media_type, drive_path)

            if media_type == 'playlist':
                playlist_path = os.path.join(drive_path, sanitize_name(info['title']))
                status.value = f"Download complete! Files saved to: {playlist_path}"
            else:
                video_path = os.path.join(drive_path, sanitize_name(info['title']))
                status.value = f"Download complete! Files saved to: {video_path}"

            progress.value = '100%'

        except Exception as e:
            status.value = f"Error: {str(e)}"
            progress.value = '0%'

# GUI Components
url_widget = widgets.Text(placeholder='Enter YouTube URL', layout={'width': '500px'})
type_widget = widgets.Dropdown(options=['video', 'playlist'], value='video')
download_btn = widgets.Button(description='Download', button_style='success')
progress = widgets.HTML(value='0%')
status = widgets.HTML(value='Status: Ready')
output = widgets.Output()

download_btn.on_click(on_button_click)

gui = widgets.VBox([
    widgets.HBox([widgets.Label('URL:'), url_widget]),
    widgets.HBox([widgets.Label('Type:'), type_widget]),
    download_btn,
    widgets.HTML('<b>Progress:</b>'),
    progress,
    widgets.HTML('<b>Status:</b>'),
    status,
    output
])

display(gui)